In [0]:
#calculando retorno diario

from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_ticker = (
    Window
    .partitionBy("Codigo")
    .orderBy("Data")
)
window_ticker

In [0]:
#ler a tabela de cotações 

fato = spark.table("mercado_financeiro.gold.fato_cotacoes")

In [0]:
#criando o fechamento anterior
from pyspark.sql.window import Window
from pyspark.sql import functions as F

fato = (
    fato
    .withColumn(
        "Fechamento_Anterior",
        F.lag("Fechamento").over(window_ticker)

    )
)

In [0]:
#calculando o retorno

fato = (
    fato
    .withColumn(
        "Retorno_Diario",
        (
            F.col("Fechamento") /
            F.col("Fechamento_Anterior")
        ) - 1
    )
)

In [0]:
#validar 

display(
    fato.select(
        "Data",
        "Codigo",
        "Fechamento",
        "Fechamento_Anterior",
        "Retorno_Diario"
    )
)

In [0]:
#coluna media movel 20 períodos

window_mm20 = (
    Window
    .partitionBy("Codigo")
    .orderBy("Data")
    .rowsBetween(-19, 0)
)

fato = (
    fato
    .withColumn(
        "Média_Movel_20",
        F.avg("Fechamento").over(window_mm20)
    )
)

In [0]:
fato.show(2)

In [0]:
#coluna media movel 50

window_mm50 = (
    Window
    .partitionBy("Codigo")
    .orderBy("Data")
    .rowsBetween(-49, 0)
)

fato = (
    fato
    .withColumn(
        "Média_Movel_50",
        F.avg("Fechamento").over(window_mm50)
    )
)

In [0]:
#coluna volatilidade vol20

window_vol20 = (
    Window
    .partitionBy("Codigo")
    .orderBy("Data")
    .rowsBetween(-19, 0 )
)

fato = (
    fato
    .withColumn(
        "volatilidade_20",
        F.stddev("Retorno_Diario").over(window_vol20))
    
)

In [0]:
fato.show(2)

In [0]:
print(type(window_ticker))

In [0]:
#coluna primeiro fechamento
#vou aproveitar a window function que eu ja tenho criado - window_ticker

fato = (
    fato
    .withColumn(
        "Primeiro_Fechamento",
        F.first("Fechamento").over(window_ticker)
    )
)





In [0]:
#calcular o retorno acumulado

fato = (
    fato
    .withColumn(
        "Retorno_acumulado",
        (F.col("Fechamento") / F.col("Primeiro_Fechamento")) -1
    )
)

In [0]:
fato.show(2)

In [0]:
#salvando como fato_cotacoes_indicadores na camada gold

(
    fato
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mercado_financeiro.gold.fato_cotacoes_indicadores")


)